# 03 — Pattern Mining

Runs n-gram motif extraction (k=3..7) on each persona's token sequence.
Significance test: chi-square of observed vs expected under independence,
thresholded by a shuffled-baseline permutation test.

Outputs `output/motifs_{persona}.json` containing:
- `top_motifs`: top 10 significant n-grams by chi2
- `motif_strength_score`: 0-100, becomes the **Consistency** stat axis

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))
from pattern_miner import compute_motifs, normalise_scores, token_entropy

OUTPUT = Path('..') / 'output'
PERSONAS = ['aisyah', 'daniel', 'mei_ling', 'hafiz']

In [ ]:
# Pass 1: compute raw chi2 sums + token entropy for all personas
raw_results: dict = {}

for persona in PERSONAS:
    with open(OUTPUT / f'tokens_{persona}.json', encoding='utf-8') as f:
        tokens = json.load(f)

    print(f'Mining {persona} ({len(tokens)} tokens)...')
    top_motifs, raw_chi2 = compute_motifs(tokens, seed=42)
    h = token_entropy(tokens)
    raw_results[persona] = {'top_motifs': top_motifs, 'raw_chi2': raw_chi2, 'entropy': h}
    print(f'  raw_chi2={raw_chi2:.1f}  entropy={h:.3f}  motifs={len(top_motifs)}')

In [ ]:
# Pass 2: normalise to 0-100 motif_strength_score
# chi2_weight=0.30: entropy carries 70% weight so Daniel (concentrated food_grab
# distribution, lower H=3.521) scores above Mei Ling (most diverse, H=3.761)
# even when both have chi2=0. Structural chi2 patterns dominate when present.
raw_chi2_map = {p: v['raw_chi2']  for p, v in raw_results.items()}
entropy_map  = {p: v['entropy']   for p, v in raw_results.items()}
normalised   = normalise_scores(raw_chi2_map, entropy_map, chi2_weight=0.30, target_max=92.0)

print('\nMotif strength scores (= Consistency axis):')
print(f'  {"Persona":<12} {"Raw chi2":>12}  {"Entropy":>8}  {"Score":>6}')
print('-' * 48)
for p in sorted(normalised, key=lambda x: -normalised[x]):
    print(f'  {p:<12} {raw_chi2_map[p]:>12.1f}  {entropy_map[p]:>8.3f}  {normalised[p]:>6.1f}')

In [ ]:
# Pass 3: write output files
for persona in PERSONAS:
    result = {
        'top_motifs': raw_results[persona]['top_motifs'],
        'motif_strength_score': normalised[persona],
    }
    out_path = OUTPUT / f'motifs_{persona}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2)

print('\nTop motifs per persona:')
for persona in PERSONAS:
    with open(OUTPUT / f'motifs_{persona}.json', encoding='utf-8') as f:
        data = json.load(f)
    print(f'\n{persona.upper()}  (score={data["motif_strength_score"]})')
    for m in data['top_motifs'][:3]:
        print(f'  [{m["k"]}-gram count={m["count"]} chi2={m["chi2"]:.0f}]  {m["motif"]}')